In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
OUTPUT = TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
# maximum-likelihood career stages from Zhang et al., Scientific productivity as a random walk
STAGES = [(0,4,'StageOne'), (5,7,'StageTwo'), (8,13,'StageThree'), (14,20,'StageFour')]

In [ ]:
ranks = pd.read_csv(OUTPUT/'rank_correlations.csv')
start_ages = {'age_0':1, 'lag_5':5}
ranks = ranks.loc[ranks.measure.eq('annual') & ranks.reference.isin(start_ages)]
ranks = ranks.assign(start_age=ranks.reference.map(start_ages))
ranks = ranks.loc[ranks.age.ge(ranks.start_age) & ranks.rho.notna()]
keys = ['source','reference']
ages = ranks.age - ranks.groupby(keys).age.transform('mean')
# OLS slope with an intercept, excluding self-correlation at age zero
groups = ranks.assign(weighted_rho=ages*ranks.rho, age_sq=ages**2).groupby(keys)
slopes = groups.weighted_rho.sum()/groups.age_sq.sum()
windows = groups.age.agg(start_age='min', end_age='max')
windows.assign(slope=slopes).to_csv(OUTPUT/'rank_persistence_slopes.csv')

In [ ]:
# annual output distances grouped by career year
distances = pd.read_csv(OUTPUT/'annual_wasserstein.csv')
stage_rows = []
for start,end,stage in STAGES:
    selected = distances.loc[distances.age.between(start,end)]
    stage_rows.append(selected.groupby('source',sort=False).wasserstein1
                      .agg(mean_wasserstein1='mean',n_years='size').reset_index()
                      .assign(stage=stage,start_age=start,end_age=end))
stage_w1 = pd.concat(stage_rows,ignore_index=True)
stage_w1.to_csv(OUTPUT/'stage_wasserstein_summary.csv',index=False)

In [ ]:
# pooled shared empirical productivity bins; require full bin coverage
dropout = pd.read_csv(OUTPUT/'dropout_by_productivity.csv').query("stage == 'pooled'")
probability = dropout.pivot(index='bin', columns='source', values='probability')
weights = dropout.loc[dropout.source.eq('empirical_available')].set_index('bin').n
errors = probability.reindex(columns=CONFIG['models']).sub(probability.empirical_available, axis=0).loc[weights.index]
dropout_rmse = (errors.pow(2).mul(weights, axis=0).sum(min_count=len(weights))/weights.sum())**.5
dropout_rmse.rename_axis('source').rename('weighted_rmse').to_csv(OUTPUT/'dropout_agreement.csv')